In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# -----------------------------
# 1. Load files
# -----------------------------
np.random.seed(42)

data_dir = Path("data/clean")
output_dir = Path("data/output")

expected = pd.read_csv(output_dir / "expected_pnl.csv")
prices = pd.read_csv(data_dir / "prices.csv")
trade_detail = pd.read_csv(output_dir / "trade_pnl_detail.csv")

# Parse dates
expected["date"] = pd.to_datetime(expected["date"])
prices["date"] = pd.to_datetime(prices["date"])
trade_detail["date"] = pd.to_datetime(trade_detail["date"])

# -----------------------------
# 2. Build base reported P&L table
# -----------------------------
# Use expected_total_pnl as the clean starting point
base = expected[["date", "instrument", "expected_total_pnl", "trade_count", "start_position", "end_position"]].copy()

# Merge in precise prices from prices.csv
base = base.merge(
    prices[["date", "instrument", "close_price", "prior_close_price"]],
    on=["date", "instrument"],
    how="left"
)

# Create row key
base["row_key"] = base["date"].dt.strftime("%Y-%m-%d") + "|" + base["instrument"]

# Start reported P&L equal to expected P&L
reported = base[["date", "instrument"]].copy()
reported["reported_pnl"] = base["expected_total_pnl"].copy()

# For quick row lookup
row_lookup = {row_key: idx for idx, row_key in enumerate(base["row_key"])}

# Track used rows so each date/instrument only gets one planted break
used_keys = set()

# Scenario log
scenario_log = []

# -----------------------------
# 3. Helper functions
# -----------------------------
def severity_bucket(abs_delta):
    if abs_delta > 750:
        return "High"
    elif abs_delta > 300:
        return "Medium"
    elif abs_delta > 150:
        return "Low"
    else:
        return "Below Threshold"

def apply_delta(row_key, exception_type, delta_pnl, note, trade_id=None,
                fee_variance=None, stale_price_used=None, position_delta=None):
    """
    Apply a P&L delta to reported_pnl for one date/instrument row
    and log the planted scenario.
    """
    idx = row_lookup[row_key]

    before = float(reported.at[idx, "reported_pnl"])
    after = before + delta_pnl
    reported.at[idx, "reported_pnl"] = after

    row = base.loc[idx]

    scenario_log.append({
        "date": row["date"].strftime("%Y-%m-%d"),
        "instrument": row["instrument"],
        "row_key": row_key,
        "exception_type": exception_type,
        "expected_pnl": round(before, 2),
        "reported_pnl_after_injection": round(after, 2),
        "delta_pnl": round(delta_pnl, 2),
        "abs_delta_pnl": round(abs(delta_pnl), 2),
        "severity": severity_bucket(abs(delta_pnl)),
        "trade_id": trade_id,
        "fee_variance": fee_variance,
        "stale_price_used": stale_price_used,
        "position_delta": position_delta,
        "note": note
    })

    used_keys.add(row_key)

def first_n_unique_rows(df, n, key_col="row_key"):
    """
    Iterates through df in order and returns up to n rows
    with unique row_key values not already used.
    """
    selected = []
    seen = set()

    for _, row in df.iterrows():
        key = row[key_col]
        if key in used_keys or key in seen:
            continue
        selected.append(row)
        seen.add(key)
        if len(selected) == n:
            break

    return selected

# -----------------------------
# 4. Prepare trade-level candidate impacts
# -----------------------------
# Merge precise close prices into trade_detail
trade_detail = trade_detail.merge(
    prices[["date", "instrument", "close_price"]],
    on=["date", "instrument"],
    how="left"
)

trade_detail["row_key"] = trade_detail["date"].dt.strftime("%Y-%m-%d") + "|" + trade_detail["instrument"]

# Signed quantity
trade_detail["signed_qty"] = np.where(
    trade_detail["side"] == "BUY",
    trade_detail["quantity"],
    -trade_detail["quantity"]
)

# Approximate same-day P&L contribution of the trade
# This is a simplified but reasonable way to simulate the effect
# of a missing or duplicated trade in the reported system.
trade_detail["trade_pnl_impact"] = (
    trade_detail["signed_qty"] * (trade_detail["close_price"] - trade_detail["trade_price"])
    - trade_detail["fee_amount"]
)

trade_detail["abs_trade_pnl_impact"] = trade_detail["trade_pnl_impact"].abs()

# Sort largest first so planted breaks are more likely to be material
trade_candidates = trade_detail.sort_values("abs_trade_pnl_impact", ascending=False).copy()

# -----------------------------
# 5. Inject duplicate booking (2 rows)
# -----------------------------
dup_rows = first_n_unique_rows(trade_candidates, n=2)

for row in dup_rows:
    delta = float(row["trade_pnl_impact"])
    apply_delta(
        row_key=row["row_key"],
        exception_type="duplicate_booking",
        delta_pnl=delta,
        trade_id=row["trade_id"],
        note="Reported system duplicated one trade booking, overstating or understating same-day P&L by the duplicated trade's contribution."
    )

# -----------------------------
# 6. Inject missing trade (2 rows)
# -----------------------------
remaining_trade_candidates = trade_candidates[~trade_candidates["row_key"].isin(used_keys)].copy()
missing_rows = first_n_unique_rows(remaining_trade_candidates, n=2)

for row in missing_rows:
    delta = -float(row["trade_pnl_impact"])
    apply_delta(
        row_key=row["row_key"],
        exception_type="missing_trade",
        delta_pnl=delta,
        trade_id=row["trade_id"],
        note="Reported system omitted one executed trade, removing that trade's same-day P&L contribution."
    )

# -----------------------------
# 7. Inject incorrect fee (2 rows)
# -----------------------------
fee_candidates = base[
    (base["trade_count"] > 0) &
    (~base["row_key"].isin(used_keys))
].copy()

# Sort by trade_count descending to make fee issues feel more plausible
fee_candidates = fee_candidates.sort_values("trade_count", ascending=False)

fee_rows = first_n_unique_rows(fee_candidates, n=2)

fee_adjustments = [175.00, 250.00]

for row, fee_adj in zip(fee_rows, fee_adjustments):
    apply_delta(
        row_key=row["row_key"],
        exception_type="incorrect_fee",
        delta_pnl=-fee_adj,
        fee_variance=fee_adj,
        note="Reported system applied an incorrect aggregated fee/commission charge, lowering reported P&L."
    )

# -----------------------------
# 8. Inject stale price (2 rows)
# -----------------------------
stale_candidates = base[
    (~base["row_key"].isin(used_keys)) &
    (base["end_position"] != 0)
].copy()

# Use precise stale price impact:
# if reported system uses prior_close instead of close, the error is:
# end_position * (prior_close - close)
stale_candidates["stale_price_delta"] = (
    stale_candidates["end_position"] *
    (stale_candidates["prior_close_price"] - stale_candidates["close_price"])
)

stale_candidates["abs_stale_price_delta"] = stale_candidates["stale_price_delta"].abs()
stale_candidates = stale_candidates.sort_values("abs_stale_price_delta", ascending=False)

stale_rows = first_n_unique_rows(stale_candidates, n=2)

for row in stale_rows:
    delta = float(row["stale_price_delta"])
    apply_delta(
        row_key=row["row_key"],
        exception_type="stale_price",
        delta_pnl=delta,
        stale_price_used=row["prior_close_price"],
        note="Reported system used prior close instead of current close to mark the end-of-day position."
    )

# -----------------------------
# 9. Inject wrong position (2 rows)
# -----------------------------
wrong_pos_candidates = base[
    (~base["row_key"].isin(used_keys)) &
    (base["end_position"] != 0)
].copy()

# For a wrong-position scenario, we simulate a reported position mismatch.
# Approximate daily P&L impact = position_delta * (close - prior_close)
wrong_pos_candidates["daily_move"] = (
    wrong_pos_candidates["close_price"] - wrong_pos_candidates["prior_close_price"]
)

wrong_pos_rows = []

for _, row in wrong_pos_candidates.iterrows():
    row_key = row["row_key"]
    daily_move = float(row["daily_move"])

    if daily_move == 0:
        continue

    chosen_position_delta = None
    chosen_delta_pnl = None

    # Try increasingly larger quantity mismatches until the impact is material
    for qty in [25000, 50000, 75000, 100000, 150000]:
        sign = np.random.choice([-1, 1])
        candidate_position_delta = sign * qty
        candidate_delta_pnl = candidate_position_delta * daily_move

        if abs(candidate_delta_pnl) >= 150:
            chosen_position_delta = candidate_position_delta
            chosen_delta_pnl = candidate_delta_pnl
            break

    if chosen_position_delta is not None and row_key not in used_keys:
        wrong_pos_rows.append({
            "row_key": row_key,
            "position_delta": chosen_position_delta,
            "delta_pnl": chosen_delta_pnl
        })

    if len(wrong_pos_rows) == 2:
        break

for row in wrong_pos_rows:
    apply_delta(
        row_key=row["row_key"],
        exception_type="wrong_position",
        delta_pnl=float(row["delta_pnl"]),
        position_delta=int(row["position_delta"]),
        note="Reported system carried an incorrect end-of-day position quantity, distorting mark-to-market P&L."
    )

# -----------------------------
# 10. Finalize outputs
# -----------------------------
reported["reported_pnl"] = reported["reported_pnl"].round(2)

scenario_log_df = pd.DataFrame(scenario_log)

# Sort outputs
reported = reported.sort_values(["date", "instrument"]).reset_index(drop=True)
scenario_log_df = scenario_log_df.sort_values(["date", "instrument"]).reset_index(drop=True)

# Save files
reported.to_csv(output_dir / "reported_pnl.csv", index=False)
scenario_log_df.to_csv(output_dir / "break_scenario_log.csv", index=False)

# -----------------------------
# 11. Quick summary prints
# -----------------------------
print("Step 5 complete.")
print(f"reported_pnl rows: {len(reported)}")
print(f"scenario log rows: {len(scenario_log_df)}")

print("\nScenario counts:")
print(scenario_log_df["exception_type"].value_counts())

print("\nSample reported_pnl:")
print(reported.head(10))

print("\nSample break_scenario_log:")
print(scenario_log_df.head(10))

Step 5 complete.
reported_pnl rows: 135
scenario log rows: 10

Scenario counts:
exception_type
incorrect_fee        2
wrong_position       2
missing_trade        2
duplicate_booking    2
stale_price          2
Name: count, dtype: int64

Sample reported_pnl:
        date instrument  reported_pnl
0 2025-01-02     AUDUSD         -8.69
1 2025-01-02     EURUSD       -269.32
2 2025-01-02     GBPUSD       -240.35
3 2025-01-03     AUDUSD       -695.18
4 2025-01-03     EURUSD          0.00
5 2025-01-03     GBPUSD        524.25
6 2025-01-06     AUDUSD        184.88
7 2025-01-06     EURUSD        -57.01
8 2025-01-06     GBPUSD       -782.78
9 2025-01-07     AUDUSD        187.60

Sample break_scenario_log:
         date instrument            row_key     exception_type  expected_pnl  \
0  2025-01-02     EURUSD  2025-01-02|EURUSD      incorrect_fee        -94.32   
1  2025-01-02     GBPUSD  2025-01-02|GBPUSD      incorrect_fee          9.65   
2  2025-01-03     AUDUSD  2025-01-03|AUDUSD     wrong_po

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# -----------------------------
# 1. Load files
# -----------------------------
data_dir = Path("data/clean")
output_dir = Path("data/output")

expected = pd.read_csv(output_dir / "expected_pnl.csv")
reported = pd.read_csv(output_dir / "reported_pnl.csv")
prices = pd.read_csv(data_dir / "prices.csv")
positions = pd.read_csv(data_dir / "positions.csv")
trade_detail = pd.read_csv(output_dir / "trade_pnl_detail.csv")

# Parse dates
for df in [expected, reported, prices, positions, trade_detail]:
    df["date"] = pd.to_datetime(df["date"])

# -----------------------------
# 2. Build base exception table
# -----------------------------
exception_df = (
    expected[[
        "date", "instrument", "expected_total_pnl", "trade_count",
        "start_position", "end_position"
    ]]
    .merge(reported, on=["date", "instrument"], how="left")
    .merge(prices[["date", "instrument", "close_price", "prior_close_price"]],
           on=["date", "instrument"], how="left")
    .merge(positions[["date", "instrument", "start_position", "end_position"]],
           on=["date", "instrument"], how="left", suffixes=("", "_positions"))
)

# Clean up duplicated columns if merge created them
if "start_position_positions" in exception_df.columns:
    exception_df.drop(columns=["start_position_positions", "end_position_positions"], inplace=True)

exception_df.rename(columns={"expected_total_pnl": "expected_pnl"}, inplace=True)

# Core variance
exception_df["pnl_variance"] = exception_df["reported_pnl"] - exception_df["expected_pnl"]
exception_df["abs_pnl_variance"] = exception_df["pnl_variance"].abs()

# Threshold logic
MATERIALITY_THRESHOLD = 150.0

def severity_bucket(x):
    if x > 750:
        return "High"
    elif x > 300:
        return "Medium"
    elif x > 150:
        return "Low"
    else:
        return "None"

exception_df["is_exception"] = exception_df["abs_pnl_variance"] > MATERIALITY_THRESHOLD
exception_df["severity"] = exception_df["abs_pnl_variance"].apply(severity_bucket)

# Status and aging
latest_date = exception_df["date"].max()

def business_day_age(start_date, end_date):
    return np.busday_count(start_date.date(), end_date.date())

exception_df["status"] = np.where(exception_df["is_exception"], "Open", "No Exception")
exception_df["age_business_days"] = exception_df["date"].apply(
    lambda d: business_day_age(d, latest_date) if pd.notnull(d) else np.nan
)

# -----------------------------
# 3. Precompute trade-level impacts
# -----------------------------
# We recreate the same-day trade contribution logic used in Step 5
trade_detail = trade_detail.merge(
    prices[["date", "instrument", "close_price"]],
    on=["date", "instrument"],
    how="left"
)

trade_detail["signed_qty"] = np.where(
    trade_detail["side"] == "BUY",
    trade_detail["quantity"],
    -trade_detail["quantity"]
)

trade_detail["trade_pnl_impact"] = (
    trade_detail["signed_qty"] * (trade_detail["close_price"] - trade_detail["trade_price"])
    - trade_detail["fee_amount"]
)

# -----------------------------
# 4. Classification helpers
# -----------------------------
TRADE_TOL = 15.0
PRICE_TOL = 15.0
LOT_INCREMENT = 25000
LOT_TOL = 5000

def classify_exception(row, trade_detail):
    """
    Returns:
        likely_cause
        cause_confidence
        matched_trade_id
        matched_trade_impact
        stale_price_candidate_delta
        implied_position_delta
        investigation_note
    """
    if not row["is_exception"]:
        return (
            "none", "none", None, None, None, None,
            "Variance below materiality threshold."
        )

    date = row["date"]
    instrument = row["instrument"]
    variance = row["pnl_variance"]
    abs_variance = row["abs_pnl_variance"]
    trade_count = row["trade_count"]
    end_position = row["end_position"]
    close_price = row["close_price"]
    prior_close = row["prior_close_price"]

    matched_trade_id = None
    matched_trade_impact = None
    stale_price_candidate_delta = None
    implied_position_delta = None

    # Same-day trades for this row
    day_trades = trade_detail[
        (trade_detail["date"] == date) &
        (trade_detail["instrument"] == instrument)
    ].copy()

    # -------------------------
    # A. Stale price check
    # -------------------------
    stale_price_candidate_delta = end_position * (prior_close - close_price)

    stale_match = abs(variance - stale_price_candidate_delta) <= PRICE_TOL

    # -------------------------
    # B. Duplicate / missing trade check
    # -------------------------
    duplicate_match = False
    missing_match = False
    duplicate_best_diff = np.inf
    missing_best_diff = np.inf
    duplicate_trade_id = None
    missing_trade_id = None
    duplicate_trade_impact = None
    missing_trade_impact = None

    if not day_trades.empty:
        # duplicate booking => reported - expected ~= + trade_impact
        day_trades["dup_diff"] = (variance - day_trades["trade_pnl_impact"]).abs()

        # missing trade => reported - expected ~= - trade_impact
        day_trades["miss_diff"] = (variance + day_trades["trade_pnl_impact"]).abs()

        dup_best = day_trades.loc[day_trades["dup_diff"].idxmin()]
        miss_best = day_trades.loc[day_trades["miss_diff"].idxmin()]

        duplicate_best_diff = dup_best["dup_diff"]
        missing_best_diff = miss_best["miss_diff"]

        if duplicate_best_diff <= TRADE_TOL:
            duplicate_match = True
            duplicate_trade_id = dup_best["trade_id"]
            duplicate_trade_impact = dup_best["trade_pnl_impact"]

        if missing_best_diff <= TRADE_TOL:
            missing_match = True
            missing_trade_id = miss_best["trade_id"]
            missing_trade_impact = miss_best["trade_pnl_impact"]

    # -------------------------
    # C. Wrong position check
    # -------------------------
    daily_move = close_price - prior_close
    wrong_position_match = False

    if abs(daily_move) > 1e-9:
        implied_position_delta = variance / daily_move
        rounded_position_delta = round(implied_position_delta / LOT_INCREMENT) * LOT_INCREMENT

        if rounded_position_delta != 0:
            reconstructed_delta = rounded_position_delta * daily_move

            # wrong_position should look like a plausible round-lot quantity mismatch
            # and should not just be the stale-price case in disguise
            if (
                abs(implied_position_delta - rounded_position_delta) <= LOT_TOL and
                abs(variance - reconstructed_delta) <= PRICE_TOL and
                abs(rounded_position_delta + end_position) > LOT_TOL
            ):
                wrong_position_match = True
                implied_position_delta = rounded_position_delta

    # -------------------------
    # D. Assign likely cause using priority
    # -------------------------
    if stale_match:
        return (
            "stale_price", "high", None, None,
            round(stale_price_candidate_delta, 2),
            None,
            "Variance closely matches using prior close instead of current close on the end-of-day position."
        )

    # If both matched, choose the closer one
    if duplicate_match and missing_match:
        if duplicate_best_diff <= missing_best_diff:
            return (
                "duplicate_booking", "high",
                duplicate_trade_id, round(duplicate_trade_impact, 2),
                round(stale_price_candidate_delta, 2),
                None,
                "Variance closely matches the P&L contribution of one duplicated same-day trade."
            )
        else:
            return (
                "missing_trade", "high",
                missing_trade_id, round(missing_trade_impact, 2),
                round(stale_price_candidate_delta, 2),
                None,
                "Variance closely matches the removal of one same-day trade's P&L contribution."
            )

    if duplicate_match:
        return (
            "duplicate_booking", "high",
            duplicate_trade_id, round(duplicate_trade_impact, 2),
            round(stale_price_candidate_delta, 2),
            None,
            "Variance closely matches the P&L contribution of one duplicated same-day trade."
        )

    if missing_match:
        return (
            "missing_trade", "high",
            missing_trade_id, round(missing_trade_impact, 2),
            round(stale_price_candidate_delta, 2),
            None,
            "Variance closely matches the removal of one same-day trade's P&L contribution."
        )

    if wrong_position_match:
        return (
            "wrong_position", "medium",
            None, None,
            round(stale_price_candidate_delta, 2),
            int(implied_position_delta),
            "Variance is consistent with a plausible round-lot end-of-day position mismatch."
        )

    # fee is the fallback after stronger rules are ruled out
    if trade_count > 0 and abs_variance <= 300:
        return (
            "incorrect_fee", "medium",
            None, None,
            round(stale_price_candidate_delta, 2),
            None,
            "Variance is moderate, trade-related, and not well explained by price or trade-booking logic; likely fee discrepancy."
        )

    return (
        "unknown", "low",
        None, None,
        round(stale_price_candidate_delta, 2),
        implied_position_delta if pd.notnull(implied_position_delta) else None,
        "Variance is material but not cleanly explained by current rule set."
    )

# -----------------------------
# 5. Apply classification
# -----------------------------
results = exception_df.apply(
    lambda row: classify_exception(row, trade_detail),
    axis=1,
    result_type="expand"
)

results.columns = [
    "likely_cause",
    "cause_confidence",
    "matched_trade_id",
    "matched_trade_impact",
    "stale_price_candidate_delta",
    "implied_position_delta",
    "investigation_note"
]

exception_df = pd.concat([exception_df, results], axis=1)

# Optional: only keep likely_cause for flagged rows
exception_df.loc[~exception_df["is_exception"], "likely_cause"] = "none"
exception_df.loc[~exception_df["is_exception"], "cause_confidence"] = "none"

# Round for readability
numeric_cols = [
    "expected_pnl", "reported_pnl", "pnl_variance", "abs_pnl_variance",
    "close_price", "prior_close_price", "stale_price_candidate_delta",
    "matched_trade_impact", "implied_position_delta"
]

for col in numeric_cols:
    if col in exception_df.columns:
        exception_df[col] = exception_df[col].round(2)

# Sort
exception_df = exception_df.sort_values(["date", "instrument"]).reset_index(drop=True)

# -----------------------------
# 6. Save exception report
# -----------------------------
exception_df.to_csv(output_dir / "exception_report.csv", index=False)

print("Step 6 complete.")
print(f"exception_report rows: {len(exception_df)}")

print("\nFlagged exception counts by likely cause:")
print(exception_df.loc[exception_df["is_exception"], "likely_cause"].value_counts())

print("\nFlagged exception sample:")
print(
    exception_df.loc[exception_df["is_exception"], [
        "date", "instrument", "expected_pnl", "reported_pnl",
        "pnl_variance", "severity", "likely_cause",
        "cause_confidence", "matched_trade_id", "implied_position_delta"
    ]].head(15)
)

Step 6 complete.
exception_report rows: 135

Flagged exception counts by likely cause:
likely_cause
incorrect_fee        2
wrong_position       2
missing_trade        2
duplicate_booking    2
stale_price          2
Name: count, dtype: int64

Flagged exception sample:
          date instrument  expected_pnl  reported_pnl  pnl_variance severity  \
1   2025-01-02     EURUSD        -94.32       -269.32       -175.00      Low   
2   2025-01-02     GBPUSD          9.65       -240.35       -250.00      Low   
3   2025-01-03     AUDUSD       -474.68       -695.18       -220.50      Low   
5   2025-01-03     GBPUSD        349.50        524.25        174.75      Low   
8   2025-01-06     GBPUSD      -1018.23       -782.78        235.45      Low   
38  2025-01-20     GBPUSD       -267.22       -630.08       -362.86   Medium   
59  2025-01-29     GBPUSD        683.42        907.27        223.85      Low   
67  2025-02-03     EURUSD       -412.72       -660.57       -247.85      Low   
105 2025-02-

In [3]:
break_log = pd.read_csv(output_dir / "break_scenario_log.csv")
break_log["date"] = pd.to_datetime(break_log["date"])

validation = (
    exception_df.loc[exception_df["is_exception"], ["date", "instrument", "likely_cause"]]
    .merge(
        break_log[["date", "instrument", "exception_type"]],
        on=["date", "instrument"],
        how="left"
    )
)

validation["match"] = validation["likely_cause"] == validation["exception_type"]

print(validation)
print("\nClassification accuracy:")
print(validation["match"].mean())

        date instrument       likely_cause     exception_type  match
0 2025-01-02     EURUSD      incorrect_fee      incorrect_fee   True
1 2025-01-02     GBPUSD      incorrect_fee      incorrect_fee   True
2 2025-01-03     AUDUSD     wrong_position     wrong_position   True
3 2025-01-03     GBPUSD     wrong_position     wrong_position   True
4 2025-01-06     GBPUSD      missing_trade      missing_trade   True
5 2025-01-20     GBPUSD  duplicate_booking  duplicate_booking   True
6 2025-01-29     GBPUSD      missing_trade      missing_trade   True
7 2025-02-03     EURUSD  duplicate_booking  duplicate_booking   True
8 2025-02-20     AUDUSD        stale_price        stale_price   True
9 2025-02-27     AUDUSD        stale_price        stale_price   True

Classification accuracy:
1.0
